In [2]:
import pyodbc
import time
import random
from datetime import datetime, timedelta
import threading

# اتصال به دیتابیس
conn = pyodbc.connect(
    'DRIVER={SQL Server};'
    'SERVER=10WKS-PISHVA;'
    'DATABASE=PEGAH;'
    'UID=rezapishva;'
    'PWD=5rdx@4rfv1355'
)
cursor = conn.cursor()

# دیکشنری مقادیر نمونه بر اساس AssetID (از فایل SAMPLE.csv استخراج شده)
# AssetID -> (Value, PersonelID نمونه)
asset_templates = {
    8341: (6.9, 83746),    # فرض: 8341 شبیه 8346 رفتار می‌کنه
    8342: (12.1, 83746),
    8343: (69.0, 83746),
    8344: (-240.0, 83746),
    8346: (6.9, 83746),
    9286: (7.2, 83746),
    9287: (1.28, 83746)
}

# لیست AssetID های مورد نظر
target_assets = [8341, 8342, 8343, 8344, 8346, 9286, 9287]

# UnitID همیشه 11
UNIT_ID = 11

# تابع تولید 7 رکورد با اختلاف TimeStamp کمتر از 1000
def generate_seven_records():
    # زمان پایه: همین الان
    base_time = datetime.now()
    base_timestamp = int(time.time())

    # اختلاف زمانی تصادفی بین 0 تا 900 ثانیه (کمتر از 1000)
    start_offset = random.randint(0, 900)
    start_timestamp = base_timestamp + start_offset
    start_datetime = base_time + timedelta(seconds=start_offset)

    records = []
    time_offset = 0

    for asset_id in target_assets:
        value, personel_id = asset_templates[asset_id]

        # زمان فعلی با افزایش تصادفی 1 تا 10 ثانیه
        current_timestamp = start_timestamp + time_offset
        current_datetime = start_datetime + timedelta(seconds=time_offset)

        record_time = current_datetime.strftime("%H:%M:%S")
        record_date = current_datetime.strftime("%Y/%m/%d")
        datetime_str = current_datetime.strftime("%Y-%m-%d %H:%M:%S.000")

        records.append((
            asset_id,
            UNIT_ID,
            value,
            record_time,
            record_date,
            personel_id,
            datetime_str,
            current_timestamp
        ))

        # افزایش تصادفی زمان (1 تا 10 ثانیه)
        time_offset += random.randint(1, 10)

    return records

# تابع اینسرت به دیتابیس
def insert_records(records):
    insert_query = """
    INSERT INTO [PEGAH].[PDA].[Periodic_Values]
    ([AssetID], [UnitID], [Value], [RecordTime], [RecordDate], [PersonelID], [DateTime], [TimeStamp])
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """

    try:
        cursor.executemany(insert_query, records)
        conn.commit()
        print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 7 رکورد با موفقیت اینسرت شد.")
    except Exception as e:
        conn.rollback()
        print(f"خطا در اینسرت: {e}")

# تابع اصلی که هر 10 دقیقه اجرا میشه
def run_periodic_insert():
    while True:
        records = generate_seven_records()
        insert_records(records)
        time.sleep(60)  # 10 دقیقه = 600 ثانیه

# اجرای تسک در پس‌زمینه
def start_scheduler():
    thread = threading.Thread(target=run_periodic_insert, daemon=True)
    thread.start()
    print("Scheduler شروع شد. هر 10 دقیقه 7 رکورد اینسرت میشه...")
    return thread

# شروع اسکریپت
if __name__ == "__main__":
    start_scheduler()
    
    # برای جلوگیری از بسته شدن اسکریپت (در محیط واقعی مثل سرور)
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print("اسکریپت متوقف شد.")
        conn.close()

Scheduler شروع شد. هر 10 دقیقه 7 رکورد اینسرت میشه...
[2025-11-12 18:38:24] 7 رکورد با موفقیت اینسرت شد.
[2025-11-12 18:39:24] 7 رکورد با موفقیت اینسرت شد.
[2025-11-12 18:40:26] 7 رکورد با موفقیت اینسرت شد.
[2025-11-12 18:41:26] 7 رکورد با موفقیت اینسرت شد.
[2025-11-12 18:42:26] 7 رکورد با موفقیت اینسرت شد.
[2025-11-12 18:43:26] 7 رکورد با موفقیت اینسرت شد.
[2025-11-12 18:44:26] 7 رکورد با موفقیت اینسرت شد.
[2025-11-12 18:45:26] 7 رکورد با موفقیت اینسرت شد.
[2025-11-12 18:46:26] 7 رکورد با موفقیت اینسرت شد.
[2025-11-12 18:47:26] 7 رکورد با موفقیت اینسرت شد.
[2025-11-12 18:48:26] 7 رکورد با موفقیت اینسرت شد.
[2025-11-12 18:49:26] 7 رکورد با موفقیت اینسرت شد.
[2025-11-12 18:50:26] 7 رکورد با موفقیت اینسرت شد.
[2025-11-12 18:51:26] 7 رکورد با موفقیت اینسرت شد.
[2025-11-12 18:52:26] 7 رکورد با موفقیت اینسرت شد.
[2025-11-12 18:53:27] 7 رکورد با موفقیت اینسرت شد.
اسکریپت متوقف شد.


Exception in thread Thread-6 (run_periodic_insert):
Traceback (most recent call last):
  File "C:\Users\pishva_r\AppData\Local\Temp\ipykernel_14460\1318771695.py", line 85, in insert_records
pyodbc.ProgrammingError: The cursor's connection has been closed.

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\pishva_r\Anaconda3\envs\test2\Lib\threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "c:\Users\pishva_r\Anaconda3\envs\test2\Lib\site-packages\ipykernel\ipkernel.py", line 761, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\pishva_r\Anaconda3\envs\test2\Lib\threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\pishva_r\AppData\Local\Temp\ipykernel_14460\1318771695.py", line 96, in run_periodic_insert
  File "C:\Users\pishva_r\AppData\Local\Temp\ipykernel_14460\1318771695.py", line 89, in insert_records
pyodbc.ProgrammingError: Attempt to u

In [2]:
df1 = pd.read_excel('output_lube_oil_g11.xlsx')

In [3]:
# انتخاب ستون‌ها برای استانداردسازی
data_to_scale = df1[['AssetID_8341', 'AssetID_8342', 'AssetID_8343', 'AssetID_8344',
       'AssetID_8346', 'AssetID_9286', 'AssetID_9287']]

# استانداردسازی داده‌ها
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data_to_scale)

# تبدیل خروجی به دیتافریم با همان نام ستون‌ها
scaled_df = pd.DataFrame(scaled_data, columns=['AssetID_8341', 'AssetID_8342', 'AssetID_8343', 'AssetID_8344',
       'AssetID_8346', 'AssetID_9286', 'AssetID_9287'])


In [4]:
scaled_df_clean = scaled_df.dropna()

In [5]:
# اجرای DBSCAN
dbscan = DBSCAN(eps=0.7, min_samples=6)
labels = dbscan.fit_predict(scaled_df_clean)

# اضافه کردن لیبل‌ها به دیتافریم
df = scaled_df_clean.copy()
df['label'] = labels

# جدا کردن داده‌های نویز و خوشه‌ها
noise_mask = df['label'] == -1
cluster_mask = df['label'] != -1

noise_points = df[noise_mask].drop(columns='label').values
cluster_points = df[cluster_mask].drop(columns='label').values

# محاسبه فاصله هر نویز از نزدیک‌ترین نقطه در خوشه‌ها
distances = pairwise_distances(noise_points, cluster_points)
min_distances = distances.min(axis=1)

# نرمال‌سازی فاصله‌ها به بازه 0 تا 1
normalized_weights = (min_distances - min_distances.min()) / (min_distances.max() - min_distances.min())

# ساخت سری وزن ناهنجاری برای همه داده‌ها
anomaly_weights = np.zeros(len(df))
anomaly_weights[noise_mask.values] = normalized_weights

# اضافه کردن به دیتافریم نهایی
df['anomaly_weight'] = anomaly_weights


In [6]:
# train_model.py
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
import joblib
import warnings

warnings.filterwarnings("ignore")

# بارگذاری داده‌ها
df1 = pd.read_excel('output_lube_oil_g11.xlsx')
selected_columns = ['AssetID_8341', 'AssetID_8342', 'AssetID_8343', 'AssetID_8344',
                    'AssetID_8346', 'AssetID_9286', 'AssetID_9287']
data_to_scale = df1[selected_columns]

# استانداردسازی
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data_to_scale)
scaled_df = pd.DataFrame(scaled_data, columns=selected_columns)
scaled_df_clean = scaled_df.dropna()

# اجرای DBSCAN
dbscan = DBSCAN(eps=0.7, min_samples=6)
labels = dbscan.fit_predict(scaled_df_clean)

# جدا کردن داده‌های نویز و خوشه‌ها
df = scaled_df_clean.copy()
df['label'] = labels
noise_mask = df['label'] == -1
cluster_mask = df['label'] != -1
noise_points = df[noise_mask].drop(columns='label').values
cluster_points = df[cluster_mask].drop(columns='label').values

# محاسبه فاصله و وزن ناهنجاری
distances = pairwise_distances(noise_points, cluster_points)
min_distances = distances.min(axis=1)
normalized_weights = (min_distances - min_distances.min()) / (min_distances.max() - min_distances.min())

# ساخت بردار وزن ناهنجاری
anomaly_weights = np.zeros(len(df))
anomaly_weights[noise_mask.values] = normalized_weights
df['anomaly_weight'] = anomaly_weights

# ذخیره مدل‌ها و داده‌های مرجع
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(dbscan, 'dbscan_model.pkl')
np.save('cluster_points.npy', cluster_points)


In [6]:
import pyodbc

# اتصال به SQL Server
conn = pyodbc.connect(
    'DRIVER={SQL Server};'
    'SERVER=10WKS-PISHVA;'
    'DATABASE=PEGAH;'
    'Trusted_Connection=yes;'
)

cursor = conn.cursor()

# لیست AssetIDهایی که می‌خوای بررسی بشن
asset_ids = [8341, 8342, 8343, 8344, 8346, 9286, 9287]

# لیست برای ذخیره مقادیر Value
values = []

# اجرای کوئری برای هر AssetID و دریافت فقط Value
for asset_id in asset_ids:
    query = f"""
        SELECT TOP 1 [Value]
        FROM [PDA].[Periodic_Values]
        WHERE UnitID = 11 AND AssetID = {asset_id}
        ORDER BY DateTime DESC
    """
    cursor.execute(query)
    row = cursor.fetchone()
    if row:
        values.append(row.Value)
    else:
        values.append(None)  # اگر رکوردی نبود، مقدار None قرار می‌گیره

# قرار دادن مقادیر در ۷ متغیر جداگانه
value_8341, value_8342, value_8343, value_8344, value_8346, value_9286, value_9287 = values

# نمایش مقادیر
print("✅ مقادیر آخرین رکوردها برای UnitID=11:")
print(f"AssetID 8341 → Value: {value_8341}")
print(f"AssetID 8342 → Value: {value_8342}")
print(f"AssetID 8343 → Value: {value_8343}")
print(f"AssetID 8344 → Value: {value_8344}")
print(f"AssetID 8346 → Value: {value_8346}")
print(f"AssetID 9286 → Value: {value_9286}")
print(f"AssetID 9287 → Value: {value_9287}")

# بستن اتصال
cursor.close()
conn.close()




import numpy as np
import pandas as pd
from sklearn.metrics import pairwise_distances
import joblib

# بارگذاری اجزای مدل
scaler = joblib.load('scaler.pkl')
dbscan = joblib.load('dbscan_model.pkl')
cluster_points = np.load('cluster_points.npy')

# تابع تشخیص ناهنجاری با آستانه وزن
def is_anomalous(input_dict, threshold=10.0):
    input_df = pd.DataFrame([input_dict])
    scaled_input = scaler.transform(input_df)
    label = dbscan.fit_predict(scaled_input)[0]

    if label != -1:
        return {'is_anomaly': False, 'anomaly_weight': 0.0}

    # محاسبه فاصله از نزدیک‌ترین خوشه
    distance = pairwise_distances(scaled_input, cluster_points).min()
    anomaly_weight = distance

    # بررسی آستانه
    is_anomaly = anomaly_weight > threshold
    return {'is_anomaly': is_anomaly, 'anomaly_weight': anomaly_weight}

# مثال استفاده
sample_input = {
    'AssetID_8341': value_8341,
    'AssetID_8342': value_8342,
    'AssetID_8343': value_8343,
    'AssetID_8344': value_8344,
    'AssetID_8346': value_8346,
    'AssetID_9286': value_9286,
    'AssetID_9287': value_9287
}

result = is_anomalous(sample_input)
print(result)



✅ مقادیر آخرین رکوردها برای UnitID=11:
AssetID 8341 → Value: 0.07
AssetID 8342 → Value: 12.1
AssetID 8343 → Value: 69.0
AssetID 8344 → Value: -240.0
AssetID 8346 → Value: 6.9
AssetID 9286 → Value: 7.2
AssetID 9287 → Value: 1.28
{'is_anomaly': False, 'anomaly_weight': 0.0}


In [1]:
import pyodbc

# اتصال به SQL Server
conn = pyodbc.connect(
    'DRIVER={SQL Server};'
    'SERVER=10WKS-PISHVA;'
    'DATABASE=PEGAH;'
    'Trusted_Connection=yes;'
)

cursor = conn.cursor()

# لیست AssetIDهایی که می‌خوای بررسی بشن
asset_ids = [8341, 8342, 8343, 8344, 8346, 9286, 9287]

# لیست برای ذخیره مقادیر Value
values = []

# اجرای کوئری برای هر AssetID و دریافت فقط Value
for asset_id in asset_ids:
    query = f"""
        SELECT TOP 1 [Value]
        FROM [PDA].[Periodic_Values]
        WHERE UnitID = 11 AND AssetID = {asset_id}
        ORDER BY DateTime DESC
    """
    cursor.execute(query)
    row = cursor.fetchone()
    if row:
        values.append(row.Value)
    else:
        values.append(None)  


value_8341, value_8342, value_8343, value_8344, value_8346, value_9286, value_9287 = values

# نمایش مقادیر
print("✅ مقادیر آخرین رکوردها برای UnitID=11:")
print(f"AssetID 8341 → Value: {value_8341}")
print(f"AssetID 8342 → Value: {value_8342}")
print(f"AssetID 8343 → Value: {value_8343}")
print(f"AssetID 8344 → Value: {value_8344}")
print(f"AssetID 8346 → Value: {value_8346}")
print(f"AssetID 9286 → Value: {value_9286}")
print(f"AssetID 9287 → Value: {value_9287}")

# بستن اتصال
cursor.close()
conn.close()




import numpy as np
import pandas as pd
from sklearn.metrics import pairwise_distances
import joblib

# بارگذاری اجزای مدل
scaler = joblib.load('scaler.pkl')
dbscan = joblib.load('dbscan_model.pkl')
cluster_points = np.load('cluster_points.npy')

# تابع تشخیص ناهنجاری با آستانه وزن
def is_anomalous(input_dict, threshold=10.0):
    input_df = pd.DataFrame([input_dict])
    scaled_input = scaler.transform(input_df)
    label = dbscan.fit_predict(scaled_input)[0]

    if label != -1:
        return {'is_anomaly': False, 'anomaly_weight': 0.0}

    # محاسبه فاصله از نزدیک‌ترین خوشه
    distance = pairwise_distances(scaled_input, cluster_points).min()
    anomaly_weight = distance

    # بررسی آستانه
    is_anomaly = anomaly_weight > threshold
    return {'is_anomaly': is_anomaly, 'anomaly_weight': anomaly_weight}

# مثال استفاده
sample_input = {
    'AssetID_8341': value_8341,
    'AssetID_8342': value_8342,
    'AssetID_8343': value_8343,
    'AssetID_8344': value_8344,
    'AssetID_8346': value_8346,
    'AssetID_9286': value_9286,
    'AssetID_9287': value_9287
}

result = is_anomalous(sample_input)
print(result)



import mysql.connector
from datetime import datetime

# اتصال به دیتابیس MySQL
conn = mysql.connector.connect(
    host='127.0.0.1',
    port=3306,
    user='root',
    password='',  
    database='dsas'
)

cursor = conn.cursor()

# مقادیر ورودی
inputs = [value_8341,value_8342,value_8343,value_8344,value_8346,value_9286,value_9287]
anomaly_weight = result['anomaly_weight']  # مقدار score
results = "Normal" if anomaly_weight < 5 else "Abnormal"
model_name = "Anomaly detection for lube oil system"
unitID = 11
system = "dbscan clustering weighted by computing distance from clusters "
score = anomaly_weight
created_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
updated_at = created_at


query = """
    INSERT INTO results_dsas_mhi_lube_oil_11 
    (inputs, results, model_name, unitID, system, score, created_at, updated_at)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""


cursor.execute(query, (
    str(inputs),  # تبدیل لیست به رشته برای ذخیره در فیلد text یا varchar
    results,
    model_name,
    unitID,
    system,
    score,
    created_at,
    updated_at
))

# ذخیره تغییرات
conn.commit()

print("✅ داده با موفقیت ثبت شد.")

# بستن اتصال
cursor.close()
conn.close()


ModuleNotFoundError: No module named 'pyodbc'

In [21]:
import pyodbc
server = '172.28.232.26'
database = 'DSAS'
username = 'mps-mapnaom\\dsasadmin'   
password = 'Dsas99@kz.net'
conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=172.28.232.26;"
    "DATABASE=DSAS;"
    "Trusted_Connection=yes;"
)

try:
    conn = pyodbc.connect(conn_str)
    print("اتصال با حساب ویندوز ادمین موفقیت‌آمیز بود!")   
    conn.close()
except Exception as e:
    print("خطا در اتصال:", e)

خطا در اتصال: ('28000', "[28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Login failed for user 'MAPNAOM-KZ\\Pishva_r'. (18456) (SQLDriverConnect); [28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Login failed for user 'MAPNAOM-KZ\\Pishva_r'. (18456)")


In [17]:
import mysql.connector

try:
    conn = mysql.connector.connect(
        host='172.28.232.27',
        user='root',
        password='',
        database='mapna_gaurd'  # اگر دیتابیس خاصی مدنظرته، نامش رو اینجا بذار
    )

    if conn.is_connected():
        print("اتصال برقرار شد ✅")
        cursor = conn.cursor()
        cursor.execute("SELECT DATABASE();")
        db = cursor.fetchone()
        print("دیتابیس فعلی:", db)

except mysql.connector.Error as err:
    print("خطا در اتصال:", err)

finally:
    if 'conn' in locals() and conn.is_connected():
        conn.close()
        print("اتصال بسته شد 🔒")


# CREATE USER 'root'@'172.28.232.164' IDENTIFIED BY '';
# GRANT ALL PRIVILEGES ON *.* TO 'root'@'172.28.232.164' WITH GRANT OPTION;
# FLUSH PRIVILEGES;






اتصال برقرار شد ✅
دیتابیس فعلی: ('mapna_gaurd',)
اتصال بسته شد 🔒


In [ ]:
# pip install mysql-connector-python
# !pip install pyodbc

In [ ]:
# pip install --upgrade pip

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----------------- ---------------------- 0.8/1.8 MB 2.4 MB/s eta 0:00:01
   ----------------------------------- ---- 1.6/1.8 MB 3.1 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 2.7 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.1.1
    Uninstalling pip-25.1.1:
      Successfully uninstalled pip-25.1.1
Note: you may need to restart the kernel to use updated packages.
